In [50]:
import pandas as pd

df = pd.read_parquet('../data/reddit_wsb_cache.parquet')

df.head()

,eastern,text,mentioned_tickers
1,2021-01-28 06:32:10-05:00,Math Professor Scott Steiner says the numbers ...,[GME]
2,2021-01-28 06:30:35-05:00,Exit the system The CEO of NASDAQ pushed to ha...,"[GME, QQQ]"
3,2021-01-28 06:28:57-05:00,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,[GME]
4,2021-01-28 06:26:56-05:00,"Not to distract from GME, just thought our AMC...","[GME, AMC]"
6,2021-01-28 06:26:27-05:00,SHORT STOCK DOESN'T HAVE AN EXPIRATION DATE He...,[T]


In [51]:
unique_tickers = sorted(set().union(*df['mentioned_tickers']))

In [52]:
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

results = []
maybe_bad_tickers = []

def get_earnings(ticker):
  td = yf.Ticker(ticker)
  actions = td.actions
  if actions is None:
    maybe_bad_tickers.append(ticker)
    return
  earnings = td.get_earnings_dates()
  if earnings is None:
    maybe_bad_tickers.append(ticker)
    return
  actions_dates = pd.to_datetime(actions.index).date
  earnings_dates = pd.to_datetime(earnings.index).date
  all_dates = pd.Series(list(actions_dates) + list(earnings_dates)).drop_duplicates().tolist()
  results.append({'ticker': ticker, 'ignoredates': all_dates})

with ThreadPoolExecutor(max_workers=8) as executor:
  executor.map(get_earnings, unique_tickers)

df = pd.DataFrame(results)

$ABMD: possibly delisted; no timezone found
ABMD: No earnings dates found, symbol may be delisted
$ANSS: possibly delisted; no timezone found
$ATVI: possibly delisted; no timezone found
ATVI: No earnings dates found, symbol may be delisted
$CTLT: possibly delisted; no timezone found
$DFS: possibly delisted; no timezone found
$DISH: possibly delisted; no timezone found
DISH: No earnings dates found, symbol may be delisted
$EXPR: possibly delisted; no timezone found
EXPR: No earnings dates found, symbol may be delisted
$FLT: possibly delisted; no timezone found
FLT: No earnings dates found, symbol may be delisted
$HES: possibly delisted; no timezone found
$MRO: possibly delisted; no timezone found
$MWRK: possibly delisted; no timezone found
MWRK: No earnings dates found, symbol may be delisted
$PBSTV: possibly delisted; no timezone found
PBSTV: No earnings dates found, symbol may be delisted
$RE: possibly delisted; no timezone found
$SPLK: possibly delisted; no timezone found
$WISH: poss

In [53]:
df.to_parquet('../data/stock_ignoredates.parquet', engine='pyarrow', compression='gzip')

In [54]:
from datetime import timedelta

reddit_posts = pd.read_parquet('../data/reddit_wsb_cache.parquet')
stock_ignore_dates = pd.read_parquet('../data/stock_ignoredates.parquet')


In [55]:
unique_tickers = sorted(set(stock_ignore_dates['ticker']))
stock_price_windows = []

for ticker in unique_tickers:
  posts_with_ticker = reddit_posts[reddit_posts['mentioned_tickers'].apply(lambda tickers: ticker in tickers)]
  if posts_with_ticker.empty:
    continue
  stock_price_windows.append({
    'ticker': ticker,
    'start': (posts_with_ticker['eastern'].min() - timedelta(days=7)).strftime('%Y-%m-%d'),
    'stop': (posts_with_ticker['eastern'].max() + timedelta(days=7)).strftime('%Y-%m-%d'),
  })

In [60]:
stock_prices = []

def get_price_data(window):
  ticker = window['ticker']
  start = window['start']
  stop = window['stop']
  
  try:
    prices = yf.download(ticker, start=start, end=stop, progress=False, auto_adjust=False)
    if prices.empty:
      print(f"No price data for {ticker} between {start} and {stop}")
      return
  except Exception as e:
    print(f"Error fetching {ticker}: {e}")
    return
  for date, price in prices['Adj Close'][ticker].items():
    stock_prices.append({
      'ticker': ticker,
      'date': date.date(),
      'price': price
    })

for w in stock_price_windows:
  get_price_data(w)

all_prices = pd.DataFrame(stock_prices)


1 Failed download:
['ANSS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


No price data for ANSS between 2021-01-30 and 2021-04-09



1 Failed download:
['CTLT']: YFTzMissingError('possibly delisted; no timezone found')


No price data for CTLT between 2021-03-10 and 2021-03-24



1 Failed download:
['DFS']: YFTzMissingError('possibly delisted; no timezone found')


No price data for DFS between 2021-05-16 and 2021-05-30



1 Failed download:
['HES']: YFTzMissingError('possibly delisted; no timezone found')


No price data for HES between 2021-01-21 and 2021-07-14



1 Failed download:
['MRO']: YFTzMissingError('possibly delisted; no timezone found')


No price data for MRO between 2021-02-05 and 2021-06-27


In [63]:
all_prices.to_parquet('../data/stock_prices.parquet', engine='pyarrow', compression='gzip')

In [64]:
print(len(stock_price_windows))
print(len(all_prices))
print(all_prices.head())

305
28941
  ticker        date      price
0    AAL  2021-01-21  15.830000
1    AAL  2021-01-22  15.820000
2    AAL  2021-01-25  15.430000
3    AAL  2021-01-26  15.530000
4    AAL  2021-01-27  16.559999
